# English-to-French ML Translator: Complete Project

This notebook trains a lightweight hybrid translator on 10,000 deterministic parallel pairs. A character n-gram SVM predicts sentence structure; constrained bilingual decoding produces a translation; nearest-neighbour retrieval handles out-of-grammar fallback.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit

ROOT=Path.cwd(); assert (ROOT/'src/translator.py').exists(); sys.path.insert(0,str(ROOT/'src'))
from translator import load_pairs, HybridTranslator
(ROOT/'artifacts').mkdir(exist_ok=True)
rows=load_pairs(ROOT/'data/english_french_pairs.tsv')
print('Parallel pairs:',len(rows),'Templates:',len({row['template_id'] for row in rows}),'Meanings:',len({row['group_id'] for row in rows}))


Parallel pairs: 10000 Templates: 30 Meanings: 2392


## 1. Group-aware split prevents register variants of one meaning crossing the boundary


In [2]:
groups=[row['group_id'] for row in rows]
train_i,test_i=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(rows,groups=groups))
train=[rows[i] for i in train_i]; test=[rows[i] for i in test_i]
assert set(row['group_id'] for row in train).isdisjoint(row['group_id'] for row in test)
print('Train:',len(train),'Test:',len(test))


Train: 8004 Test: 1996


## 2. Fit and evaluate


In [3]:
model=HybridTranslator().fit(train)
metrics=model.evaluate(test)
print(json.dumps(metrics.as_dict(),indent=2))
model.save(ROOT/'artifacts/translator.joblib')
(ROOT/'artifacts/metrics.json').write_text(json.dumps(metrics.as_dict(),indent=2))


{
  "pairs": 1996,
  "exact_match": 0.9814629258517034,
  "template_accuracy": 0.9894789579158316,
  "bleu4": 0.9810758441634614,
  "chrf3": 0.9905064345375749
}
161


In [4]:
metric_df=pd.Series(metrics.as_dict()).drop('pairs')
metric_df.plot(kind='bar',color='#6a4c93',title='Held-out translation metrics',ylim=(0,1.05))
plt.tight_layout(); plt.savefig(ROOT/'artifacts/translation_metrics.png',dpi=160); plt.show()


## 3. Inspect inference


In [5]:
examples=['i like the book','we visit paris today','the museum is open','they travel by train','can you help me']
pd.DataFrame([{'english':text,**model.translate(text)} for text in examples])


                english  ... retrieved_pair_id
0       i like the book  ...        pair_02421
1  we visit paris today  ...        pair_09554
2    the museum is open  ...        pair_07340
3  they travel by train  ...        pair_02361
4       can you help me  ...        pair_04802

[5 rows x 5 columns]


## Deployment and honesty boundary

Run `python app.py --model artifacts/translator.joblib --port 8002`. This is a strong reproducible ML systems project, but it is a constrained-domain hybrid translator trained on synthetic data. Do not call its high held-out score open-domain neural-translation performance.
